In [0]:
%sql
SET TIME ZONE 'UTC';

CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE TABLE IF NOT EXISTS workspace.silver.day_ahead_price (
  bzn                STRING    NOT NULL,
  delivery_date      DATE      NOT NULL,
  delivery_start_utc TIMESTAMP NOT NULL,
  delivery_end_utc   TIMESTAMP NOT NULL,
  resolution         STRING,
  price              DOUBLE,
  currency           STRING,
  price_unit         STRING,
  is_filled          BOOLEAN   COMMENT 'true = slot omitted by ENTSO-E (A03), price carried forward',
  _source_file       STRING,
  _file_modified_at  TIMESTAMP COMMENT 'bronze file version this day was built from',
  _silver_updated_at TIMESTAMP
)
COMMENT 'Day-ahead prices: latest revision, one row per zone and delivery slot';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW changed_days AS
WITH latest_bronze AS (
  SELECT bzn, delivery_date, MAX(_file_modified_at) AS latest_version
  FROM workspace.bronze.entsoe_day_ahead_prices
  GROUP BY bzn, delivery_date
),
silver_versions AS (
  SELECT bzn, delivery_date, MAX(_file_modified_at) AS silver_version
  FROM workspace.silver.day_ahead_price
  GROUP BY bzn, delivery_date
)
SELECT b.bzn, b.delivery_date, b.latest_version
FROM latest_bronze b
LEFT JOIN silver_versions s USING (bzn, delivery_date)
WHERE s.silver_version IS NULL                 -- day not in silver yet
   OR b.latest_version > s.silver_version;     -- newer revision arrived

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW latest_points AS
WITH latest_version AS (
  SELECT p.*
  FROM workspace.bronze.entsoe_day_ahead_prices p
  JOIN changed_days c
    ON  p.bzn = c.bzn
    AND p.delivery_date = c.delivery_date
    AND p._file_modified_at = c.latest_version
),
finest_resolution AS (
  SELECT *
  FROM latest_version
  QUALIFY DENSE_RANK() OVER (
    PARTITION BY bzn, delivery_date
    ORDER BY CAST(regexp_extract(resolution, 'PT([0-9]+)M', 1) AS INT)
  ) = 1
)
SELECT *
FROM finest_resolution
WHERE delivery_date = to_date(from_utc_timestamp(delivery_start_utc, 'Europe/Berlin'))
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY bzn, delivery_start_utc ORDER BY _ingested_at DESC
) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_new AS
WITH day_meta AS (
  SELECT
    bzn,
    delivery_date,
    FIRST(resolution)        AS resolution,
    CAST(regexp_extract(FIRST(resolution), 'PT([0-9]+)M', 1) AS INT) AS step_min,
    FIRST(currency)          AS currency,
    FIRST(price_unit)        AS price_unit,
    FIRST(_source_file)      AS _source_file,
    MAX(_file_modified_at)   AS _file_modified_at
  FROM latest_points
  GROUP BY bzn, delivery_date
),
grid AS (
  SELECT d.*, slot AS delivery_start_utc
  FROM day_meta d
  LATERAL VIEW explode(
    sequence(
      to_utc_timestamp(CAST(d.delivery_date AS TIMESTAMP), 'Europe/Berlin'),
      to_utc_timestamp(CAST(date_add(d.delivery_date, 1) AS TIMESTAMP), 'Europe/Berlin')
        - INTERVAL '1' MINUTE * d.step_min,
      INTERVAL '1' MINUTE * d.step_min
    )
  ) s AS slot
)
SELECT
  g.bzn,
  g.delivery_date,
  g.delivery_start_utc,
  g.delivery_start_utc + INTERVAL '1' MINUTE * g.step_min AS delivery_end_utc,
  g.resolution,
  LAST(p.price, true) OVER (
    PARTITION BY g.bzn, g.delivery_date
    ORDER BY g.delivery_start_utc
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  )                       AS price,
  p.price IS NULL         AS is_filled,
  g.currency,
  g.price_unit,
  g._source_file,
  g._file_modified_at,
  current_timestamp()     AS _silver_updated_at
FROM grid g
LEFT JOIN latest_points p
  ON  p.bzn = g.bzn
  AND p.delivery_start_utc = g.delivery_start_utc;

In [0]:
%sql
MERGE INTO workspace.silver.day_ahead_price AS t
USING silver_new AS s
  ON  t.bzn = s.bzn
  AND t.delivery_start_utc = s.delivery_start_utc
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%python
checks = {
    "null prices or keys": """
        SELECT COUNT(*) FROM workspace.silver.day_ahead_price
        WHERE price IS NULL OR bzn IS NULL OR delivery_start_utc IS NULL""",
    "duplicate slots": """
        SELECT COUNT(*) FROM (
          SELECT bzn, delivery_start_utc FROM workspace.silver.day_ahead_price
          GROUP BY ALL HAVING COUNT(*) > 1)""",
    "unexpected slots per day": """
        SELECT COUNT(*) FROM (
          SELECT bzn, delivery_date, resolution, COUNT(*) AS n
          FROM workspace.silver.day_ahead_price GROUP BY ALL)
        WHERE NOT (
             (resolution = 'PT15M' AND n IN (92, 96, 100))
          OR (resolution = 'PT60M' AND n IN (23, 24, 25)))""",
    "days in bronze missing from silver": """
        SELECT COUNT(*) FROM (
          SELECT DISTINCT bzn, delivery_date FROM workspace.bronze.entsoe_day_ahead_prices
          EXCEPT
          SELECT DISTINCT bzn, delivery_date FROM workspace.silver.day_ahead_price)""",
}

warnings = {
    "prices outside -500..4000 EUR/MWh": """
        SELECT COUNT(*) FROM workspace.silver.day_ahead_price
        WHERE price < -500 OR price > 4000""",
}

failures = []
for name, sql in checks.items():
    n = spark.sql(sql).first()[0]
    print(f"{'OK  ' if n == 0 else 'FAIL'} {name}: {n}")
    if n > 0:
        failures.append(f"{name} ({n})")

for name, sql in warnings.items():
    n = spark.sql(sql).first()[0]
    print(f"{'OK  ' if n == 0 else 'WARN'} {name}: {n}")

if failures:
    raise Exception("Silver quality checks failed: " + ", ".join(failures))